In [ ]:
import pandas as pd
import numpy as np

from config import MY_TEAM_NAME, LEAGUE_ID, YEAR, FOLDER
from espn_pull import fetch_espn_data, fetch_espn_free_agents
from fangraphs_pull import fetch_fangraphs_projections

TOP_N_FA = 200

print(f"--- STARTING REST-OF-SEASON (ROS) ANALYSIS FOR {MY_TEAM_NAME} ---")
print("\n[Updating Data Pipelines...]")

# A. Pull FanGraphs Projections (steamer_ros; change proj_type as needed)
df_hitters_proj, df_pitchers_proj, _ = fetch_fangraphs_projections("steamer_ros")

# B. Pull ESPN Rosters and Standings
df_rosters, df_current = fetch_espn_data(LEAGUE_ID, YEAR)

# C. Pull the REAL ESPN Free Agent List
df_fa = fetch_espn_free_agents(LEAGUE_ID, YEAR)

print("[Data Pipelines Updated Successfully]\n")

# AUTO-SAVE TO CSV
print(f"Saving latest data to: {FOLDER}")
df_hitters_proj.to_csv(FOLDER / 'Fangraphs_Hitter_steamer_ros.csv', index=False)
df_pitchers_proj.to_csv(FOLDER / 'Fangraphs_Pitcher_steamer_ros.csv', index=False)
df_rosters.to_csv(FOLDER / 'espn_current_rosters.csv', index=False)
df_current.to_csv(FOLDER / 'current_team_stats.csv', index=False)
df_fa.to_csv(FOLDER / 'espn_free_agents.csv', index=False)

print("✓ All local files synchronized. Proceeding to Analysis...")

In [ ]:
from analysis import aggressive_clean, clean_projections, get_league_stats, is_eligible, get_impact_string

# ==========================================
# 2. DATA LOADING & CLEANING
# ==========================================
try:
    df_rosters      = pd.read_csv(FOLDER / 'espn_current_rosters.csv')
    df_hitters_raw  = pd.read_csv(FOLDER / 'Fangraphs_Hitter_steamer_ros.csv')
    df_pitchers_raw = pd.read_csv(FOLDER / 'Fangraphs_Pitcher_steamer_ros.csv')
    df_current      = pd.read_csv(FOLDER / 'current_team_stats.csv')
    df_espn_fa      = pd.read_csv(FOLDER / 'espn_free_agents.csv')
    print("✓ Success: Projections, Rosters, and Current YTD Stats loaded.")
except Exception as e:
    print(f"X Error: {e}")
    raise

df_rosters['Team'] = df_rosters['Team'].astype(str).str.strip()
df_current['Team'] = df_current['Team'].astype(str).str.strip()
df_rosters['Clean_Name'] = df_rosters['Player'].apply(aggressive_clean)
df_espn_fa['Clean_Name'] = df_espn_fa['Player'].apply(aggressive_clean)

df_hitters, df_pitchers = clean_projections(df_hitters_raw, df_pitchers_raw)
active_roster_full = df_rosters.copy()

print(f"ℹ Total Active Players Loaded: {len(active_roster_full)}")

In [ ]:
# ==========================================
# 3. CORE CALCULATION ENGINE
# ==========================================
# get_league_stats, is_eligible, get_impact_string, aggressive_clean, and
# clean_projections are all imported from analysis.py in Cell 2.
# No local definitions needed here.

In [ ]:
# ==========================================
# 5. PHASE 1: END OF SEASON STANDINGS
# ==========================================
baseline_stats = get_league_stats(active_roster_full, df_hitters, df_pitchers, df_current)

display_cols = [
    'Team', 'Total_Points', 'R_total', 'HR_total', 'RBI_total',
    'SB_total', 'OPS_total', 'IP_total', 'QS_total', 'SV_total',
    'ERA_total', 'WHIP_total'
]

standings = baseline_stats[display_cols].sort_values(by='Total_Points', ascending=False).reset_index(drop=True)
standings.index += 1

int_cols = ['R_total', 'HR_total', 'RBI_total', 'SB_total', 'IP_total', 'QS_total', 'SV_total']
standings[int_cols] = standings[int_cols].round(0).astype(int)

print("\n--- 1. PROJECTED END OF SEASON STANDINGS (YTD + ROS) ---")
standings_clean = standings.rename(columns={c: c.replace('_total', '') for c in standings.columns})
display(standings_clean.style.format({
    'Total_Points': '{:.1f}',
    'OPS':  '{:.3f}',
    'ERA':  '{:.2f}',
    'WHIP': '{:.2f}',
}))

In [ ]:
# ==========================================
# 6. PHASE 2: OPTIMIZED SWAPS (STRICT LEGAL)
# ==========================================

# --- 1. INITIALIZATION ---
my_players = active_roster_full[active_roster_full['Team'] == MY_TEAM_NAME].copy()

df_espn_fa_clean = (df_espn_fa
                    .drop_duplicates(subset=['Clean_Name'], keep='first')
                    .rename(columns={'Positions': 'ESPN_Positions', 'Player': 'ESPN_Player'}))

top_fa_h = pd.merge(df_espn_fa_clean, df_hitters, on='Clean_Name', how='inner').head(100)
top_fa_p = pd.merge(df_espn_fa_clean, df_pitchers, on='Clean_Name', how='inner').head(100)

base_stats      = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]
hitter_name_set = set(df_hitters['Clean_Name'])

print(f"Analyzing {len(my_players)} spots for strict legal swaps...")

h_results, p_results = [], []

# --- 2. THE SIMULATION ---
for _, drop in my_players.iterrows():
    is_hitter    = drop['Clean_Name'] in hitter_name_set
    fa_pool      = top_fa_h if is_hitter else top_fa_p
    eligible_fas = fa_pool[fa_pool['ESPN_Positions'].apply(
        lambda pos: is_eligible(pos, drop['Lineup_Slot'])
    )]

    for _, add in eligible_fas.iterrows():
        df_temp = active_roster_full[
            active_roster_full['Clean_Name'] != drop['Clean_Name']
        ].copy()
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  add['Clean_Name'],
            'Status':      'Rostered',
            'Lineup_Slot': drop['Lineup_Slot'],
        }])
        df_temp    = pd.concat([df_temp, new_row], ignore_index=True)
        sim_stats  = get_league_stats(df_temp, df_hitters, df_pitchers, df_current).set_index('Team').loc[MY_TEAM_NAME]
        point_gain = sim_stats['Total_Points'] - base_stats['Total_Points']

        if point_gain > 0:
            res = {
                'Add':      add['ESPN_Player'],
                'Drop':     drop['Player'],
                'Slot':     drop['Lineup_Slot'],
                'Net Gain': round(point_gain, 1),
                'Details':  get_impact_string(base_stats, sim_stats, baseline_stats),
            }
            if is_hitter: h_results.append(res)
            else:         p_results.append(res)

# --- 3. DISPLAY ---
print("\n--- 2. PROFITABLE LEGAL HITTER SWAPS ---")
if h_results:
    display(pd.DataFrame(h_results).sort_values(by='Net Gain', ascending=False).head(10))
else:
    print("No legal hitter swaps found that improve your total Roto points.")

print("\n--- 3. PROFITABLE LEGAL PITCHER SWAPS ---")
if p_results:
    display(pd.DataFrame(p_results).sort_values(by='Net Gain', ascending=False).head(10))
else:
    print("No legal pitcher swaps found that improve your total Roto points.")

In [ ]:
# ==========================================
# 7. PHASE 3: MANUAL TRADE & SWAP EVALUATOR
# ==========================================

# --- ENTER YOUR PROPOSED MOVES HERE ---
PLAYERS_TO_ACQUIRE       = ["Brendan Donovan"]
PLAYERS_TO_DROP_OR_TRADE = ["Jorge Polanco"]
TRADE_PARTNER            = None  # Team name string for a trade, or None for waiver pickup

print(f"\n--- EVALUATING CUSTOM SWAP ---")
print(f"Acquiring: {', '.join(PLAYERS_TO_ACQUIRE)}")
print(f"Shipping Out: {', '.join(PLAYERS_TO_DROP_OR_TRADE)}")
if TRADE_PARTNER:
    print(f"Trade Partner: {TRADE_PARTNER}")

acquire_cleaned = [aggressive_clean(p) for p in PLAYERS_TO_ACQUIRE]
drop_cleaned    = [aggressive_clean(p) for p in PLAYERS_TO_DROP_OR_TRADE]
df_sim          = active_roster_full.copy()

for raw, cleaned in zip(PLAYERS_TO_DROP_OR_TRADE, drop_cleaned):
    mask = df_sim['Clean_Name'] == cleaned
    if mask.any():
        if TRADE_PARTNER:
            df_sim.loc[mask, 'Team'] = TRADE_PARTNER
        else:
            df_sim = df_sim[~mask]
    else:
        print(f"⚠️  Warning: '{raw}' not found on your active roster (checked as '{cleaned}').")

for raw, cleaned in zip(PLAYERS_TO_ACQUIRE, acquire_cleaned):
    mask = df_sim['Clean_Name'] == cleaned
    if mask.any():
        df_sim.loc[mask, 'Team'] = MY_TEAM_NAME
    else:
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  cleaned,
            'Status':      'Rostered',
            'Lineup_Slot': 'BE',
        }])
        df_sim = pd.concat([df_sim, new_row], ignore_index=True)

sim_stats    = get_league_stats(df_sim, df_hitters, df_pitchers, df_current)
base_my_team = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]
sim_my_team  = sim_stats.set_index('Team').loc[MY_TEAM_NAME]
gain         = sim_my_team['Total_Points'] - base_my_team['Total_Points']
impact_str   = get_impact_string(base_my_team, sim_my_team, baseline_stats)

print(f"\n[ RESULTS FOR {MY_TEAM_NAME} ]")
print(f"Net Point Change: {'+' if gain > 0 else ''}{gain:.1f} Points")
print(f"Category Shifts:  {impact_str}")

print("\n--- NEW STANDINGS PREVIEW (TOP 5) ---")
display_cols = [
    'Team', 'Total_Points', 'R_total', 'HR_total', 'RBI_total',
    'SB_total', 'OPS_total', 'IP_total', 'QS_total', 'SV_total',
    'ERA_total', 'WHIP_total',
]
sim_standings = (sim_stats[display_cols]
                 .sort_values(by='Total_Points', ascending=False)
                 .reset_index(drop=True))
sim_standings.index += 1
int_cols = ['R_total', 'HR_total', 'RBI_total', 'SB_total', 'IP_total', 'QS_total', 'SV_total']
sim_standings[int_cols] = sim_standings[int_cols].round(0).astype(int)
sim_standings_clean = sim_standings.rename(columns={c: c.replace('_total', '') for c in sim_standings.columns})
display(sim_standings_clean.head(5).style.format({
    'Total_Points': '{:.1f}', 'OPS': '{:.3f}', 'ERA': '{:.2f}', 'WHIP': '{:.2f}',
}))

In [ ]:
# ==========================================
# 8. PHASE 4: EMPTY SLOT FILLER (BEST FA BY POSITION)
# ==========================================
TARGET_POSITION        = "RP"   # e.g. 'OF', 'SP', '1B', 'C', 'RP'
MAX_CANDIDATES_TO_TEST = 50
TOP_N_RESULTS          = 10

print(f"\n--- FINDING BEST AVAILABLE '{TARGET_POSITION}' FOR {MY_TEAM_NAME} ---")

is_pitcher = TARGET_POSITION in {'SP', 'RP', 'P'}

available_fas = df_espn_fa[
    df_espn_fa['Positions'].astype(str).str.contains(
        rf'\b{TARGET_POSITION}\b', regex=True, na=False
    )
].copy()
fa_names = available_fas['Clean_Name'].tolist()

if not fa_names:
    print(f"⚠️  No free agents with '{TARGET_POSITION}' eligibility found in espn_free_agents.csv.")
else:
    proj_pool  = df_pitchers if is_pitcher else df_hitters
    sort_col   = 'IP' if is_pitcher else 'PA'
    candidates = (proj_pool[proj_pool['Clean_Name'].isin(fa_names)]
                  .sort_values(by=sort_col, ascending=False)
                  .head(MAX_CANDIDATES_TO_TEST))

    print(f"Simulating adding the top {len(candidates)} projected {TARGET_POSITION}s to your roster...")

    fill_results = []
    base_my_team = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]

    for _, add_player in candidates.iterrows():
        df_temp = active_roster_full.copy()
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  add_player['Clean_Name'],
            'Status':      'Rostered',
            'Lineup_Slot': TARGET_POSITION,
        }])
        df_temp   = pd.concat([df_temp, new_row], ignore_index=True)
        new_stats = get_league_stats(df_temp, df_hitters, df_pitchers, df_current).set_index('Team').loc[MY_TEAM_NAME]
        gain      = new_stats['Total_Points'] - base_my_team['Total_Points']
        fill_results.append({
            'Add':      add_player.get('Player', add_player['Clean_Name']),
            'Net Gain': round(gain, 1),
            'Details':  get_impact_string(base_my_team, new_stats, baseline_stats),
        })

    if fill_results:
        display(pd.DataFrame(fill_results)
                .sort_values(by='Net Gain', ascending=False)
                .head(TOP_N_RESULTS))
    else:
        print(f"⚠️  No {TARGET_POSITION}s with matching projections found.")